# 01 — Data load & preparation

Assumes `00_config.ipynb` has been run.

In [2]:
import json
from datasets import load_dataset
ds = load_dataset("trivia_qa", "rc", split="validation")
df = ds.to_pandas()
df = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print("Loaded rows:", len(df))
print(df.columns)

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import re, unicodedata, string,json
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

def extract_answer_text(ans):
    if isinstance(ans, dict):
        val = ans.get("value", "")
        aliases = ans.get("aliases", [])
        nor_val = ans.get("normalized_value", "")
        nor_aliases = ans.get("normalized_aliases", "")
        all_vals = [val] + [a for a in aliases if a and a != val]
        return " , ".join(sorted(set(map(str, all_vals))))
    elif isinstance(ans, list):
        texts = []
        for item in ans:
            if isinstance(item, dict):
                texts.append(item.get("value", ""))
                texts.extend(item.get("aliases", []))
                texts.append(item.get("normalized_value", ""))
                texts.append(item.get("normalized_aliases", ""))
        return " , ".join(sorted(set(filter(None, texts))))
    else:
        return str(ans) if ans else ""
    
def extract_wiki_context(entitypage):
    import numpy as np
    
    def clean_text(text):
        """Remove HTML tags, brackets, punctuation, and special characters."""
        if not isinstance(text, str):
            text = str(text)
        # Remove HTML tags (if any)
        text = re.sub(r"<.*?>", " ", text)
        # Remove brackets and their contents like [text], (text), {text}
        text = re.sub(r"\[.*?\]|\(.*?\)|\{.*?\}", " ", text)
        # Remove punctuation and special characters (keep letters, digits, and spaces)
        text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
        # Replace multiple spaces with a single space
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def words_from(x):
        """Flatten strings/lists/ndarrays (even nested) into a list of words."""
        if x is None:
            return []
        if isinstance(x, np.ndarray):
            return words_from(x.tolist())
        if isinstance(x, (list, tuple, set)):
            out = []
            for e in x:
                out.extend(words_from(e))
            return out
        return clean_text(str(x)).split()

    def content(x):
        w = words_from(x)
        return " ".join(w)

    # Case 1: a single entity page dict with arrays for each field
    if isinstance(entitypage, dict):
        return content(entitypage.get("wiki_context", []))

    # Case 2: a list of such dicts -> merge all wiki_contexts, then truncate
    if isinstance(entitypage, list):
        all_words = []
        for item in entitypage:
            if isinstance(item, dict):
                all_words += words_from(item.get("wiki_context", []))
            else:
                all_words += words_from(item)
        return " ".join(all_words)

    # Fallback
    return content(entitypage)

assert "question" in df.columns, "df must have a 'question' column"
if "question_id" in df.columns:
    df["question_id"] = df["question_id"].astype("string")
else:
    # Fall back to index, but keep it as a string ID
    df["question_id"] = pd.Series(df.index, index=df.index).astype("Int64").astype("string")

df["question_clean"] = df["question"].astype(str).str.strip()
df["answer_text"]    = df["answer"].map(extract_answer_text)
df["ctx"] = df["entity_pages"].map(extract_wiki_context)


df_q_eval = df[["question_id", "question_clean", "answer_text"]].copy()

df_q_eval = df_q_eval.drop_duplicates(subset=["question_id"]).reset_index(drop=True)

df_q_eval.to_parquet(Q_PATH, index=False)

print(df_q_eval.head(1))


In [ ]:
if "ctx" in df.columns:
    ctx_col_name = "ctx"
else:
    raise ValueError("No context column found.")


non_empty = (df[ctx_col_name].astype(str).str.strip() != "").sum()
print(f"Context column: '{ctx_col_name}' | non-empty rows: {non_empty} / {len(df)}")
df = df[df["ctx"].astype(str).str.strip() != ""].reset_index(drop=True)
print(f"After removing empty context rows: {len(df)} rows remain.")